# Phase 4 â€” Meta-Learner Training & Evaluation

Trains two meta-learners (kNN, MLP) on each meta-feature representation and evaluates
with Leave-One-Out cross-validation (LOO-CV, n=51).

**Primary metric**: Top-1 accuracy â€” did the predicted best method actually rank first?  
**Secondary metric**: LSE-MAE â€” when predicting all 6 LSE values, mean absolute error  
**Baselines**:
- Lower bound: always predict k-means (most common, 14/51 = 27%)
- Upper bound: oracle (always correct, 100%)

**Ablation**: compares Option A (hand-crafted) vs B (autoencoder) vs C (dict learning)  
**Output**: saved models in `outputs/models/`

In [19]:
import os, sys, warnings, pickle
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.metrics import confusion_matrix, classification_report

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if os.path.join(ROOT, 'src') not in sys.path:
    sys.path.insert(0, os.path.join(ROOT, 'src'))

from meta_learner import (
    extract_Xy_clf, extract_Xy_reg,
    loo_classify, loo_regress,
    baseline_always, oracle_expected_lse,
    report_clf_results, LSE_COLS, METHOD_NAMES,
    build_classifier_candidates, build_regressor_candidates,
)

META_DIR   = os.path.join(ROOT, 'data', 'meta_table')
MODELS_DIR = os.path.join(ROOT, 'outputs', 'models')
FIGS_DIR   = os.path.join(ROOT, 'outputs', 'figures')
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(FIGS_DIR, exist_ok=True)

SEED = 42
np.random.seed(SEED)
print('Imports OK')

Imports OK


In [20]:
# â”€â”€ Load the three feature tables â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
df_a = pd.read_csv(os.path.join(META_DIR, 'meta_training_optA.csv'))
df_b = pd.read_csv(os.path.join(META_DIR, 'meta_training_optB.csv'))
df_c = pd.read_csv(os.path.join(META_DIR, 'meta_training_optC.csv'))

# Drop rows where best_method is NaN (fully failed datasets)
df_a = df_a.dropna(subset=['best_method']).reset_index(drop=True)
df_b = df_b.dropna(subset=['best_method']).reset_index(drop=True)
df_c = df_c.dropna(subset=['best_method']).reset_index(drop=True)

print(f'Option A: {df_a.shape}  Option B: {df_b.shape}  Option C: {df_c.shape}')
print(f'Best method distribution (Option A):')
print(df_a['best_method'].value_counts().to_string())

Option A: (78, 27)  Option B: (78, 49)  Option C: (78, 49)
Best method distribution (Option A):
best_method
gmm          26
dbscan       21
kmeans       14
autoenc       9
agg           5
dictlearn     3


In [21]:
# â”€â”€ Baselines â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
y_true = df_a['best_method'].values

baseline_kmeans = baseline_always('kmeans', y_true)
baseline_dbscan = baseline_always('dbscan', y_true)  # most frequent in our data
oracle_acc      = 1.0
oracle_lse      = oracle_expected_lse(df_a)

# Expected LSE if always picking k-means (regardless of whether it's best)
always_kmeans_lse = df_a['LSE_kmeans'].mean()

print('=== Baselines ===')
print(f'  Always k-means accuracy   : {baseline_kmeans:.3f}  (expected LSE: {always_kmeans_lse:.3f})')
print(f'  Always dbscan  accuracy   : {baseline_dbscan:.3f}')
print(f'  Oracle accuracy           : {oracle_acc:.3f}  (expected LSE: {oracle_lse:.3f})')

=== Baselines ===
  Always k-means accuracy   : 0.179  (expected LSE: 0.618)
  Always dbscan  accuracy   : 0.269
  Oracle accuracy           : 1.000  (expected LSE: 0.767)


## kNN â€” k Sweep on Option A

In [22]:
# -- Find best k via LOO-CV on Option A ----------------------------------
X_a, y_a, feat_cols_a, ids_a = extract_Xy_clf(df_a)

k_candidates = [1, 3, 5, 7, 9, 11, 13]
k_results = {}

for k in k_candidates:
    pipe = build_classifier_candidates(best_k=k, random_state=SEED)['kNN']
    res = loo_classify(pipe, X_a, y_a)
    k_results[k] = res['accuracy']
    print(f'  k={k:2d}  accuracy={res["accuracy"]:.3f}')

best_k = max(k_results, key=k_results.get)
print(f'\nBest k = {best_k}  (accuracy={k_results[best_k]:.3f})')

  k= 1  accuracy=0.564
  k= 3  accuracy=0.397
  k= 5  accuracy=0.410
  k= 7  accuracy=0.372
  k= 9  accuracy=0.359
  k=11  accuracy=0.385
  k=13  accuracy=0.372

Best k = 1  (accuracy=0.564)


## Classification LOO-CV â€” All Models Ã— All Options

In [23]:
# -- Define model candidates ----------------------------------------------
clf_candidates = build_classifier_candidates(best_k=best_k, random_state=SEED)
print('Classifier candidates:')
for name in clf_candidates:
    print(f'  - {name}')

Classifier candidates:
  - kNN
  - LogReg
  - SVC-RBF
  - ExtraTrees
  - RF
  - MLP


In [24]:
# -- Run classification LOO-CV for all options and all model candidates ---
options = {
    'A_handcrafted': df_a,
    'B_autoencoder': df_b,
    'C_dictlearn'  : df_c,
}

clf_results = {}

for opt_name, df_opt in options.items():
    X_opt, y_opt, _, _ = extract_Xy_clf(df_opt)
    for model_name, pipe in clf_candidates.items():
        key = f'{opt_name} / {model_name}'
        print(f'Running {key} ...')
        res = loo_classify(pipe, X_opt, y_opt)
        clf_results[key] = res
        report_clf_results(key, res)

print('\nDone.')

Running A_handcrafted / kNN ...
  A_handcrafted / kNN                       Top-1 acc = 0.564  pred_dist={'dbscan': 19, 'agg': 6, 'dictlearn': 3, 'gmm': 30, 'autoenc': 9, 'kmeans': 11}
Running A_handcrafted / LogReg ...
  A_handcrafted / LogReg                    Top-1 acc = 0.410  pred_dist={'gmm': 19, 'agg': 11, 'dictlearn': 9, 'dbscan': 19, 'kmeans': 11, 'autoenc': 9}
Running A_handcrafted / SVC-RBF ...
  A_handcrafted / SVC-RBF                   Top-1 acc = 0.385  pred_dist={'gmm': 20, 'agg': 12, 'dictlearn': 7, 'dbscan': 18, 'kmeans': 13, 'autoenc': 8}
Running A_handcrafted / ExtraTrees ...
  A_handcrafted / ExtraTrees                Top-1 acc = 0.577  pred_dist={'gmm': 31, 'agg': 4, 'dictlearn': 5, 'kmeans': 11, 'autoenc': 6, 'dbscan': 21}
Running A_handcrafted / RF ...
  A_handcrafted / RF                        Top-1 acc = 0.577  pred_dist={'gmm': 30, 'dbscan': 23, 'autoenc': 6, 'kmeans': 10, 'dictlearn': 4, 'agg': 5}
Running A_handcrafted / MLP ...
  A_handcrafted / MLP       

In [25]:
# -- Summary table ---------------------------------------------------------
rows = []
for key, res in clf_results.items():
    opt, model = key.split(' / ')
    rows.append({'Option': opt, 'Model': model, 'Top-1 Accuracy': round(res['accuracy'], 3)})

rows.append({'Option': '-', 'Model': 'Always k-means', 'Top-1 Accuracy': round(baseline_kmeans, 3)})
rows.append({'Option': '-', 'Model': 'Always dbscan',  'Top-1 Accuracy': round(baseline_dbscan, 3)})
rows.append({'Option': '-', 'Model': 'Oracle (upper)', 'Top-1 Accuracy': 1.000})

summary_df = pd.DataFrame(rows)
print('=== Classification LOO-CV Results ===')
print(summary_df.sort_values('Top-1 Accuracy', ascending=False).to_string(index=False))

print('\n=== Best model per feature representation ===')
for opt_name in options:
    ranked = sorted(
        [(model_name, clf_results[f'{opt_name} / {model_name}']['accuracy']) for model_name in clf_candidates],
        key=lambda x: x[1],
        reverse=True,
    )
    print(f'  {opt_name:15s}  best={ranked[0][0]:10s}  acc={ranked[0][1]:.3f}')

=== Classification LOO-CV Results ===
       Option          Model  Top-1 Accuracy
            - Oracle (upper)           1.000
A_handcrafted     ExtraTrees           0.577
A_handcrafted             RF           0.577
A_handcrafted            MLP           0.577
A_handcrafted            kNN           0.564
B_autoencoder             RF           0.526
B_autoencoder            kNN           0.526
  C_dictlearn     ExtraTrees           0.500
B_autoencoder            MLP           0.487
  C_dictlearn            MLP           0.462
  C_dictlearn            kNN           0.462
B_autoencoder     ExtraTrees           0.462
  C_dictlearn             RF           0.449
A_handcrafted         LogReg           0.410
A_handcrafted        SVC-RBF           0.385
  C_dictlearn         LogReg           0.372
B_autoencoder        SVC-RBF           0.282
B_autoencoder         LogReg           0.282
            -  Always dbscan           0.269
  C_dictlearn        SVC-RBF           0.179
            - Alw

## Detailed Analysis â€” Best Classifier (Option A)

In [26]:
# Pick the best model on Option A to inspect in detail
option_a_results = [
    (model_name, clf_results[f'A_handcrafted / {model_name}'])
    for model_name in clf_candidates
]
best_a_name, best_a_res = max(option_a_results, key=lambda x: x[1]['accuracy'])

print(f'=== Best on Option A: {best_a_name} (acc={best_a_res["accuracy"]:.3f}) ===')
print('\nClassification report:')
print(classification_report(best_a_res['trues'], best_a_res['preds'], zero_division=0))

print('\nConfusion matrix (rows=true, cols=pred):')
labels = sorted(df_a['best_method'].unique())
cm = confusion_matrix(best_a_res['trues'], best_a_res['preds'], labels=labels)
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
print(cm_df.to_string())

=== Best on Option A: ExtraTrees (acc=0.577) ===

Classification report:
              precision    recall  f1-score   support

         agg       0.00      0.00      0.00         5
     autoenc       0.83      0.56      0.67         9
      dbscan       0.76      0.76      0.76        21
   dictlearn       0.00      0.00      0.00         3
         gmm       0.68      0.81      0.74        26
      kmeans       0.27      0.21      0.24        14

    accuracy                           0.58        78
   macro avg       0.42      0.39      0.40        78
weighted avg       0.58      0.58      0.57        78


Confusion matrix (rows=true, cols=pred):
           agg  autoenc  dbscan  dictlearn  gmm  kmeans
agg          0        0       0          2    0       3
autoenc      1        5       2          0    1       0
dbscan       0        0      16          1    2       2
dictlearn    1        0       1          0    1       0
gmm          0        0       1          1   21       3
kmeans

In [27]:
# â”€â”€ Which datasets were mis-predicted? â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
misses = [
    {
        'dataset_id': ids_a[i],
        'true':       best_a_res['trues'][i],
        'predicted':  best_a_res['preds'][i],
        'LSE_true_method':  df_a.loc[i, f'LSE_{best_a_res["trues"][i]}'],
        'LSE_pred_method':  df_a.loc[i, f'LSE_{best_a_res["preds"][i]}'],
    }
    for i in range(len(ids_a))
    if not best_a_res['correct_mask'][i]
]

miss_df = pd.DataFrame(misses)
if len(miss_df) > 0:
    miss_df['LSE_cost'] = (miss_df['LSE_true_method'] - miss_df['LSE_pred_method']).round(3)
    print(f'=== {len(miss_df)} mis-predictions ===')
    print(miss_df.to_string(index=False))
    print(f'\nMean LSE cost of errors: {miss_df["LSE_cost"].mean():.3f}')
    print(f'(= how much LSE is lost by picking the wrong method)')
else:
    print('Perfect prediction â€” no errors!')

=== 33 mis-predictions ===
 dataset_id      true predicted  LSE_true_method  LSE_pred_method  LSE_cost
      41004    kmeans       gmm           0.6899           0.6170     0.073
       4153    kmeans       agg           0.7143           0.6000     0.114
       1465       agg dictlearn           0.5833           0.3333     0.250
      46879    kmeans       gmm           0.8200           0.8200     0.000
      42186       gmm    kmeans           1.0370           0.8889     0.148
         62    kmeans dictlearn           1.0000           0.5000     0.500
         22       agg    kmeans           0.6400           0.4667     0.173
      45688    dbscan dictlearn           1.0000           0.9167     0.083
         11       gmm dictlearn           0.7732           0.5155     0.258
       1551       agg dictlearn           0.6667           0.2917     0.375
      46382   autoenc       gmm           0.4074           0.3889     0.018
        377       agg    kmeans           0.7000           0.

## Regression Variant â€” Predict All 6 LSE Values

In [28]:
reg_candidates = build_regressor_candidates(best_k=best_k, random_state=SEED)
print('Regression candidates:')
for name in reg_candidates:
    print(f'  - {name}')

Regression candidates:
  - kNN
  - RF
  - ExtraTrees
  - MLP


In [29]:
# -- Regression LOO-CV on Option A only (main approach) -------------------
X_a_reg, Y_a_reg, _, _ = extract_Xy_reg(df_a)

reg_results = {}
for model_name, pipe in reg_candidates.items():
    print(f'Running regression {model_name} ...')
    res = loo_regress(pipe, X_a_reg, Y_a_reg, df_a)
    reg_results[model_name] = res
    print(f'  MAE mean: {res["mae_mean"]:.4f}')
    print(f'  MAE per method: ' + '  '.join(
        f'{m}={v:.3f}' for m, v in zip(METHOD_NAMES, res['mae_per_col'])
    ))
    print(f'  Argmax accuracy (predict best via argmax of predicted LSE): {res["argmax_accuracy"]:.3f}')
    print(f'  Mean expected LSE of predicted method: {res["expected_lse"]:.3f}')
    print()

print(f'Oracle expected LSE (upper bound): {oracle_lse:.3f}')
print(f'Always k-means expected LSE:       {always_kmeans_lse:.3f}')

Running regression kNN ...
  MAE mean: 0.1257
  MAE per method: kmeans=0.124  dbscan=0.126  agg=0.125  gmm=0.126  autoenc=0.123  dictlearn=0.129
  Argmax accuracy (predict best via argmax of predicted LSE): 0.564
  Mean expected LSE of predicted method: 0.701

Running regression RF ...
  MAE mean: 0.1219
  MAE per method: kmeans=0.121  dbscan=0.135  agg=0.116  gmm=0.122  autoenc=0.123  dictlearn=0.114
  Argmax accuracy (predict best via argmax of predicted LSE): 0.487
  Mean expected LSE of predicted method: 0.720

Running regression ExtraTrees ...
  MAE mean: 0.1039
  MAE per method: kmeans=0.105  dbscan=0.109  agg=0.101  gmm=0.107  autoenc=0.101  dictlearn=0.100
  Argmax accuracy (predict best via argmax of predicted LSE): 0.474
  Mean expected LSE of predicted method: 0.716

Running regression MLP ...
  MAE mean: 0.1506
  MAE per method: kmeans=0.151  dbscan=0.166  agg=0.137  gmm=0.147  autoenc=0.158  dictlearn=0.145
  Argmax accuracy (predict best via argmax of predicted LSE): 0.32

## Ablation Summary â€” Option A vs B vs C

In [30]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

abl_rows = []
for opt_name in options:
    for model_name in clf_candidates:
        key = f'{opt_name} / {model_name}'
        abl_rows.append({
            'Option': opt_name,
            'Model': model_name,
            'Accuracy': clf_results[key]['accuracy'],
        })

abl_df = pd.DataFrame(abl_rows)
print('=== Classification accuracy by option/model ===')
pivot = abl_df.pivot(index='Option', columns='Model', values='Accuracy').round(3)
print(pivot.to_string())

best_rows = []
for opt_name in options:
    ranked = sorted(
        [(model_name, clf_results[f'{opt_name} / {model_name}']['accuracy']) for model_name in clf_candidates],
        key=lambda x: x[1],
        reverse=True,
    )
    best_rows.append({'Option': opt_name, 'Model': ranked[0][0], 'Accuracy': ranked[0][1]})

best_df = pd.DataFrame(best_rows)
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['steelblue', 'coral', 'darkseagreen']
bars = ax.bar(best_df['Option'], best_df['Accuracy'], color=colors)
ax.axhline(baseline_kmeans, ls='--', color='gray', label=f'Always k-means ({baseline_kmeans:.2f})')
ax.axhline(oracle_acc, ls=':', color='black', label='Oracle (1.0)')
ax.set_ylabel('Top-1 Accuracy (LOO-CV)')
ax.set_title('Best meta-learner per feature representation')
ax.set_ylim(0, 1.05)
for bar, model_name in zip(bars, best_df['Model']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02, model_name,
            ha='center', va='bottom', fontsize=9)
ax.legend(loc='lower right')
plt.tight_layout()
fig_path = os.path.join(FIGS_DIR, 'ablation_accuracy.png')
fig.savefig(fig_path, dpi=120)
plt.close()
print(f'Figure saved -> {fig_path}')

=== Classification accuracy by option/model ===
Model          ExtraTrees  LogReg    MLP     RF  SVC-RBF    kNN
Option                                                         
A_handcrafted       0.577   0.410  0.577  0.577    0.385  0.564
B_autoencoder       0.462   0.282  0.487  0.526    0.282  0.526
C_dictlearn         0.500   0.372  0.462  0.449    0.179  0.462
Figure saved -> d:\Desktop\Spring 2026 - M1\ML\project\ml-pseudo-label-meta-learning\outputs\figures\ablation_accuracy.png


## Save Best Models

In [31]:
# -- Train final models on full dataset (no held-out fold) ----------------

best_clf_name = max(
    clf_candidates,
    key=lambda name: clf_results[f'A_handcrafted / {name}']['accuracy'],
)
best_clf_pipe = clf_candidates[best_clf_name]

final_clf = clone(best_clf_pipe)
final_clf.fit(X_a, y_a)

clf_path = os.path.join(MODELS_DIR, 'meta_clf_optA.pkl')
with open(clf_path, 'wb') as f:
    pickle.dump({'pipeline': final_clf, 'feature_cols': feat_cols_a}, f)
print(f'Classifier saved -> {clf_path}  ({best_clf_name})')

best_reg_name = min(reg_results, key=lambda name: reg_results[name]['mae_mean'])
best_reg_pipe = reg_candidates[best_reg_name]

final_reg = clone(best_reg_pipe)
valid_mask = ~np.isnan(Y_a_reg).any(axis=1)
final_reg.fit(X_a[valid_mask], Y_a_reg[valid_mask])

reg_path = os.path.join(MODELS_DIR, 'meta_reg_optA.pkl')
with open(reg_path, 'wb') as f:
    pickle.dump({
        'pipeline': final_reg,
        'feature_cols': feat_cols_a,
        'lse_cols': LSE_COLS,
        'method_names': METHOD_NAMES,
    }, f)
print(f'Regressor  saved -> {reg_path}  ({best_reg_name})')

for letter, full_key_prefix, df_opt in [
    ('B', 'B_autoencoder', df_b),
    ('C', 'C_dictlearn', df_c),
]:
    X_opt, y_opt, feat_opt, _ = extract_Xy_clf(df_opt)
    best_name_opt = max(
        clf_candidates,
        key=lambda name: clf_results[f'{full_key_prefix} / {name}']['accuracy'],
    )
    best_pipe_opt = clf_candidates[best_name_opt]
    final_clf_opt = clone(best_pipe_opt)
    final_clf_opt.fit(X_opt, y_opt)
    path_opt = os.path.join(MODELS_DIR, f'meta_clf_opt{letter}.pkl')
    with open(path_opt, 'wb') as f:
        pickle.dump({'pipeline': final_clf_opt, 'feature_cols': feat_opt}, f)
    print(f'Saved opt{letter} clf -> {path_opt}  ({best_name_opt})')

Classifier saved -> d:\Desktop\Spring 2026 - M1\ML\project\ml-pseudo-label-meta-learning\outputs\models\meta_clf_optA.pkl  (ExtraTrees)
Regressor  saved -> d:\Desktop\Spring 2026 - M1\ML\project\ml-pseudo-label-meta-learning\outputs\models\meta_reg_optA.pkl  (ExtraTrees)
Saved optB clf -> d:\Desktop\Spring 2026 - M1\ML\project\ml-pseudo-label-meta-learning\outputs\models\meta_clf_optB.pkl  (kNN)
Saved optC clf -> d:\Desktop\Spring 2026 - M1\ML\project\ml-pseudo-label-meta-learning\outputs\models\meta_clf_optC.pkl  (ExtraTrees)


## Final Summary

In [32]:
print('=' * 60)
print('PHASE 4 SUMMARY')
print('=' * 60)
print(f'  n_datasets        : {len(df_a)}')
print(f'  Best k (kNN)      : {best_k}')
print()
print('  CLASSIFICATION (LOO-CV Top-1 Accuracy)')
print(f'  Lower bound       : {baseline_kmeans:.3f}  (always k-means)')
print(f'  Upper bound       : {oracle_acc:.3f}  (oracle)')
for opt_name in options:
    ranked = sorted(
        [(model_name, clf_results[f'{opt_name} / {model_name}']['accuracy']) for model_name in clf_candidates],
        key=lambda x: x[1],
        reverse=True,
    )
    score_str = ', '.join(f'{name}={score:.3f}' for name, score in ranked)
    print(f'  {opt_name:15s}: {score_str}')
print()
print('  REGRESSION (LOO-CV MAE + argmax accuracy, Option A)')
for model_name, res in reg_results.items():
    print(f'  {model_name:10s}  MAE={res["mae_mean"]:.4f}  '
          f'argmax_acc={res["argmax_accuracy"]:.3f}  '
          f'expected_LSE={res["expected_lse"]:.3f}')
print()
print(f'  Selected classifier: {best_clf_name}')
print(f'  Selected regressor : {best_reg_name}')
print(f'  Models saved to: {MODELS_DIR}')
print()
print('Phase 4 complete. Ready for Phase 5 (SHAP analysis).')

PHASE 4 SUMMARY
  n_datasets        : 78
  Best k (kNN)      : 1

  CLASSIFICATION (LOO-CV Top-1 Accuracy)
  Lower bound       : 0.179  (always k-means)
  Upper bound       : 1.000  (oracle)
  A_handcrafted  : ExtraTrees=0.577, RF=0.577, MLP=0.577, kNN=0.564, LogReg=0.410, SVC-RBF=0.385
  B_autoencoder  : kNN=0.526, RF=0.526, MLP=0.487, ExtraTrees=0.462, LogReg=0.282, SVC-RBF=0.282
  C_dictlearn    : ExtraTrees=0.500, kNN=0.462, MLP=0.462, RF=0.449, LogReg=0.372, SVC-RBF=0.179

  REGRESSION (LOO-CV MAE + argmax accuracy, Option A)
  kNN         MAE=0.1257  argmax_acc=0.564  expected_LSE=0.701
  RF          MAE=0.1219  argmax_acc=0.487  expected_LSE=0.720
  ExtraTrees  MAE=0.1039  argmax_acc=0.474  expected_LSE=0.716
  MLP         MAE=0.1506  argmax_acc=0.321  expected_LSE=0.675

  Selected classifier: ExtraTrees
  Selected regressor : ExtraTrees
  Models saved to: d:\Desktop\Spring 2026 - M1\ML\project\ml-pseudo-label-meta-learning\outputs\models

Phase 4 complete. Ready for Phase 5 (S